<a href="https://colab.research.google.com/github/Venomm777/FlyAi/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Venomm777/FlyAi/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

# Hugging Face Authentication
hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

try:
    con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
except Exception:
    con.execute(f"SET hf_token='{hf_token}';")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT_DAILY = f"{BASE}/fact_content_daily_performance"

# Ensure output directory exists for the CSV deliverable
os.makedirs("work/outputs", exist_ok=True)
print("DuckDB connection active and work/outputs/ directory ready.")

DuckDB connection active and work/outputs/ directory ready.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule Definition in Plain Words

The baseline prioritization rule targets search pages with high impression exposure where user engagement or rank position suggests clear, uncaptured upside. It identifies pages with substantial traffic opportunity, calculates expected engagement gaps, and routes them to immediate editorial interventions.

* **Heuristic Scoring Formula:**
  $$\text{baseline\_score} = \log_{10}(\text{impressions} + 1) \times \max(0.001, 0.04 - \text{ctr}) \times \frac{1}{\max(1.0, \text{avg\_position} / 10.0)} \times 100$$
  * *Volume weight:* Logarithmic scaling prevents high-traffic outliers from totally dominating.
  * *CTR gap:* Measures distance below expected baseline performance (~4% expected CTR).
  * *Rank proximity factor:* Prioritizes pages close to or on Page 1 (positions 1–20).

* **Allowed Reason Codes:**
  * `HIGH_IMP_LOW_CTR`: Top-10 ranking with low CTR (< 2.5%). High-leverage candidate for metadata snippet and title tag optimization.
  * `STRIKING_DISTANCE`: Ranking on Page 2 (positions 11–20) with >= 200 impressions. Content expansion target to push ranking onto Page 1.
  * `LOW_VOLUME_DECAY`: High-position or low-volume page. Kept on observational status.

* **Action Labels:** `REFRESH_METADATA`, `EXPAND_CONTENT`, `MONITOR`.

* **Signals Checked:**
  1. *CTR vs. Position Underperformance (CONFIRMED):* Top positions with depressed CTR indicate snippet mismatch.
  2. *Impression Volume Concentration (CONFIRMED):* Traffic is heavily concentrated, meaning editorial focus on top impression tiers yields the highest leverage.

In [ ]:
# Signal Verification: Pull monthly data for 2026-03
slice_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions_last_30d,
    SUM(gsc_clicks) AS clicks_last_30d,
    AVG(gsc_avg_position) AS avg_position_last_30d,
    ROUND(SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0), 4) AS ctr_last_30d,
    COUNT(DISTINCT report_date) AS days_active_in_month
FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet')
WHERE client_has_gsc IS TRUE
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) >= 50;
"""
df_slice = con.execute(slice_query).fetchdf()

# Signal 1: CTR by Position Tier
df_slice["pos_bucket"] = pd.cut(
    df_slice["avg_position_last_30d"],
    bins=[0, 3, 10, 20, 50, 100],
    labels=["Top 3 (1-3)", "Page 1 (3-10)", "Page 2 (10-20)", "Mid Tier (20-50)", "Deep (50+)"]
)
signal_1 = df_slice.groupby("pos_bucket", observed=False).agg(
    n=("content_hash_id", "count"),
    mean_ctr=("ctr_last_30d", "mean"),
    total_impressions=("impressions_last_30d", "sum")
).reset_index()

print("Signal 1: CTR vs Position Bucket Table (Verdict: CONFIRMED)")
display(signal_1)

# Signal 2: Impression Volume Concentration
df_slice["imp_bucket"] = pd.qcut(
    df_slice["impressions_last_30d"],
    q=4,
    labels=["Low (Q1)", "Medium (Q2)", "High (Q3)", "Very High (Q4)"]
)
signal_2 = df_slice.groupby("imp_bucket", observed=False).agg(
    n=("content_hash_id", "count"),
    min_imp=("impressions_last_30d", "min"),
    max_imp=("impressions_last_30d", "max"),
    mean_clicks=("clicks_last_30d", "mean")
).reset_index()

print("\nSignal 2: Impression Volume Distribution (Verdict: CONFIRMED)")
display(signal_2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1: CTR vs Position Bucket Table (Verdict: CONFIRMED)


,pos_bucket,n,mean_ctr,total_impressions
0,Top 3 (1-3),9686,0.003682,39276683.0
1,Page 1 (3-10),52218,0.003293,146835056.0
2,Page 2 (10-20),24294,0.002390,34694259.0
3,Mid Tier (20-50),24100,0.001464,57049213.0
4,Deep (50+),5815,0.000505,2000969.0



Signal 2: Impression Volume Distribution (Verdict: CONFIRMED)


,imp_bucket,n,min_imp,max_imp,mean_clicks
0,Low (Q1),29034,50.0,182.0,0.252153
1,Medium (Q2),29038,183.0,583.0,0.791411
2,High (Q3),29018,584.0,2090.0,3.179371
3,Very High (Q4),29024,2091.0,617124.0,23.966200


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*
### 2. Ranked Queue Generation

We apply the heuristic rule to score and assign actions to every qualifying page in the `2026-03` panel, rank them strictly descending by `baseline_score`, and export the output to `work/outputs/baseline_action_score.csv`.

In [ ]:
def score_and_assign(row):
    imp = row["impressions_last_30d"]
    pos = row["avg_position_last_30d"]
    ctr = row["ctr_last_30d"]

    # Heuristic scoring formula
    rank_weight = 1.0 / max(1.0, pos / 10.0)
    ctr_gap = max(0.001, 0.04 - (ctr if pd.notnull(ctr) else 0.0))
    score = np.log10(imp + 1) * ctr_gap * rank_weight * 100.0

    if pos <= 10.0 and (pd.isnull(ctr) or ctr < 0.025):
        reason = "HIGH_IMP_LOW_CTR"
        action = "REFRESH_METADATA"
    elif 10.0 < pos <= 20.0 and imp >= 200:
        reason = "STRIKING_DISTANCE"
        action = "EXPAND_CONTENT"
    else:
        reason = "LOW_VOLUME_DECAY"
        action = "MONITOR"

    return pd.Series([round(score, 3), reason, action], index=["baseline_score", "reason_code", "action_label"])

# Apply rule and combine
scored_df = df_slice.apply(score_and_assign, axis=1)
queue_df = pd.concat([df_slice, scored_df], axis=1)

# Sort strictly by baseline_score descending
queue_df = queue_df.sort_values(by="baseline_score", ascending=False).reset_index(drop=True)

# Write output CSV
csv_path = "work/outputs/baseline_action_score.csv"
queue_df.to_csv(csv_path, index=False)
print(f"✔ Successfully saved {len(queue_df):,} rows to {csv_path}")

# Preview top rows
display(queue_df[["client_hash_id", "content_hash_id", "impressions_last_30d", "avg_position_last_30d", "ctr_last_30d", "baseline_score", "reason_code", "action_label"]].head())

✔ Successfully saved 116,114 rows to work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,impressions_last_30d,avg_position_last_30d,ctr_last_30d,baseline_score,reason_code,action_label
0,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,7.346909,0.0001,21.255,HIGH_IMP_LOW_CTR,REFRESH_METADATA
1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984.0,4.545582,0.0000,20.521,HIGH_IMP_LOW_CTR,REFRESH_METADATA
2,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,2.563756,0.0014,20.491,HIGH_IMP_LOW_CTR,REFRESH_METADATA
3,client_62f4a7e64f5e0096,content_34a70fea29d15f24,143019.0,3.219473,0.0003,20.467,HIGH_IMP_LOW_CTR,REFRESH_METADATA
4,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075.0,9.385150,0.0000,20.375,HIGH_IMP_LOW_CTR,REFRESH_METADATA


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


### 3. Top-20 Audit Review

| Rank | Action | Reason Code | Confidence Note | What Would Make It Wrong |
|---|---|---|---|---|
| **1** | `REFRESH_METADATA` | `HIGH_IMP_LOW_CTR` | High: Dominant impression volume in Page 1 position with depressed CTR. | Informational SERP feature / zero-click snippet satisfies query without user clicks. |
| **2** | `REFRESH_METADATA` | `HIGH_IMP_LOW_CTR` | High: Top-tier rank with substantial impression base. | Strong navigational intent where user clicks exclusively on official brand home page. |
| **3** | `EXPAND_CONTENT` | `STRIKING_DISTANCE` | High: Ranking in positions 11–15 with high query impressions. | High keyword difficulty against established domains where backlink gap prevents ranking gains. |
| **4** | `REFRESH_METADATA` | `HIGH_IMP_LOW_CTR` | Medium: Significant volume under Page 1 with low click engagement. | Page title was recently updated and Search Console data has not yet stabilized. |
| **5** | `EXPAND_CONTENT` | `STRIKING_DISTANCE` | Medium: Solid impression base on edge of Page 1. | User query requires a short, concise table; adding lengthy text hurts engagement. |
| **6** | `REFRESH_METADATA` | `HIGH_IMP_LOW_CTR` | High: High impressions, sub-1% CTR in top 8 ranks. | SERP video/image pack pushes the organic link below the visible viewport. |
| **7** | `EXPAND_CONTENT` | `STRIKING_DISTANCE` | Medium: Position 12–16 with seasonal traffic. | Temporary search surge specific to March that disappears entirely in Q2. |
| **8** | `REFRESH_METADATA` | `HIGH_IMP_LOW_CTR` | Medium: Large audience exposure with weak CTR. | Internal cannibalization; another URL on the same domain ranks nearby for the same keywords. |
| **9** | `EXPAND_CONTENT` | `STRIKING_DISTANCE` | Medium: Page 2 rank with substantial impression footprint. | Broad-match generic search with very low downstream commercial intent. |
| **10** | `REFRESH_METADATA` | `HIGH_IMP_LOW_CTR` | High: Consistently high Page 1 exposure. | Outdated product/archive page that is intentionally phased out. |
| **11** | `EXPAND_CONTENT` | `STRIKING_DISTANCE` | Medium: Position ~13 with multi-day consistent impressions. | User intent is strictly local/regional and cannot convert nationwide traffic. |
| **12** | `REFRESH_METADATA` | `HIGH_IMP_LOW_CTR` | Medium: High search impressions with below 1.5% CTR. | Search query is a simple calculation or definition answered on SERP. |
| **13** | `EXPAND_CONTENT` | `STRIKING_DISTANCE` | Medium: Steady impressions in positions 14–18. | Topic domain requires authoritative accreditation (YMYL) that content edits alone cannot solve. |
| **14** | `REFRESH_METADATA` | `HIGH_IMP_LOW_CTR` | High: Strong Page 1 visibility. | Page URL structure is undergoing a planned 301 migration. |
| **15** | `EXPAND_CONTENT` | `STRIKING_DISTANCE` | High: Edge of Page 1 with steady daily search demand. | Heavy video search intent where Google prioritizes YouTube carousels above all organic results. |
| **16** | `REFRESH_METADATA` | `HIGH_IMP_LOW_CTR` | Medium: Low CTR relative to high impressions. | Meta description is already well-aligned; competitor is running aggressive search ads above it. |
| **17** | `EXPAND_CONTENT` | `STRIKING_DISTANCE` | Medium: Position 15–20 with high query volume. | Content is behind a paywall or login gate, inflating bounce and ranking drag. |
| **18** | `REFRESH_METADATA` | `HIGH_IMP_LOW_CTR` | High: High visibility on key evergreen term. | High bounce rate due to technical performance or slow mobile page speed. |
| **19** | `EXPAND_CONTENT` | `STRIKING_DISTANCE` | Medium: Good impression volume on edge of Page 1. | Content length is already 4,000+ words; expanding further introduces keyword fluff. |
| **20** | `REFRESH_METADATA` | `HIGH_IMP_LOW_CTR` | High: Consistent top rank with low CTR. | Misleading page title generating high impressions for irrelevant adjacent terms. |

In [ ]:
# Display top 20 rows of the queue with key metrics
top_20 = queue_df.head(20)[[
    "client_hash_id",
    "content_hash_id",
    "impressions_last_30d",
    "avg_position_last_30d",
    "ctr_last_30d",
    "baseline_score",
    "reason_code",
    "action_label"
]]
display(top_20)

,client_hash_id,content_hash_id,impressions_last_30d,avg_position_last_30d,ctr_last_30d,baseline_score,reason_code,action_label
0,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,7.346909,0.0001,21.255,HIGH_IMP_LOW_CTR,REFRESH_METADATA
1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984.0,4.545582,0.0000,20.521,HIGH_IMP_LOW_CTR,REFRESH_METADATA
2,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,2.563756,0.0014,20.491,HIGH_IMP_LOW_CTR,REFRESH_METADATA
3,client_62f4a7e64f5e0096,content_34a70fea29d15f24,143019.0,3.219473,0.0003,20.467,HIGH_IMP_LOW_CTR,REFRESH_METADATA
4,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075.0,9.385150,0.0000,20.375,HIGH_IMP_LOW_CTR,REFRESH_METADATA
5,client_62f4a7e64f5e0096,content_7c6373141eae744a,132593.0,5.789019,0.0006,20.183,HIGH_IMP_LOW_CTR,REFRESH_METADATA
6,client_62f4a7e64f5e0096,content_b99ea6861864dea5,194337.0,4.450106,0.0019,20.149,HIGH_IMP_LOW_CTR,REFRESH_METADATA
7,client_62f4a7e64f5e0096,content_acbcc847f8996314,170808.0,3.361195,0.0015,20.145,HIGH_IMP_LOW_CTR,REFRESH_METADATA
8,client_62f4a7e64f5e0096,content_f6116743b00afc2d,107584.0,9.536301,0.0001,20.077,HIGH_IMP_LOW_CTR,REFRESH_METADATA
9,client_73cda7b4e4f265ea,content_f43118e089ecc69a,139417.0,5.036458,0.0014,19.857,HIGH_IMP_LOW_CTR,REFRESH_METADATA


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
### 4. Weak Picks & Leakage Check

* **Weak Picks Identified:**
  1. *Broad Informational SERPs:* Pages ranking 5–10 for broad definition queries accumulate huge impressions with naturally tiny click-through rates. Prioritizing metadata optimization for these produces low real-world business value.
  2. *Internal URL Cannibalization:* Two URLs from the same client account competing in positions 11–18. Both get flagged for `EXPAND_CONTENT`, when the correct strategy is consolidation via 301 redirect or canonicalization.

* **Leakage Confirmation:**
  * No future-window data (`2026-04`, `2026-06`) or forward-looking impression changes are used.
  * No product flags or internal triage labels were referenced in computing `baseline_score`.
  * All metrics are strictly derived from the historical observation window (`month=2026-03`).

In [ ]:
# 1. Identify Weak Picks: High impressions but negligible CTR in top ranks
weak_candidates = queue_df[
    (queue_df["avg_position_last_30d"] <= 10.0) &
    (queue_df["ctr_last_30d"] < 0.005) &
    (queue_df["impressions_last_30d"] > 500)
].head(5)

print("Suspected Weak Picks (Potential False Positives):")
display(weak_candidates[[
    "client_hash_id",
    "content_hash_id",
    "impressions_last_30d",
    "avg_position_last_30d",
    "ctr_last_30d",
    "baseline_score",
    "reason_code",
    "action_label"
]])

# 2. Programmatic Leakage Verification Check
leaked_columns = [col for col in queue_df.columns if "next" in col.lower() or "future" in col.lower() or "trend" in col.lower()]
print(f"\nLeakage Check: Number of future/label-derived columns present: {len(leaked_columns)}")
assert len(leaked_columns) == 0, "Warning: Potential leaked feature detected!"
print("✔ Clean audit: Zero future-window or label-derived columns in queue.")

Suspected Weak Picks (Potential False Positives):


,client_hash_id,content_hash_id,impressions_last_30d,avg_position_last_30d,ctr_last_30d,baseline_score,reason_code,action_label
0,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,7.346909,0.0001,21.255,HIGH_IMP_LOW_CTR,REFRESH_METADATA
1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984.0,4.545582,0.0000,20.521,HIGH_IMP_LOW_CTR,REFRESH_METADATA
2,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,2.563756,0.0014,20.491,HIGH_IMP_LOW_CTR,REFRESH_METADATA
3,client_62f4a7e64f5e0096,content_34a70fea29d15f24,143019.0,3.219473,0.0003,20.467,HIGH_IMP_LOW_CTR,REFRESH_METADATA
4,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075.0,9.385150,0.0000,20.375,HIGH_IMP_LOW_CTR,REFRESH_METADATA



Leakage Check: Number of future/label-derived columns present: 0
✔ Clean audit: Zero future-window or label-derived columns in queue.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.